# Gemini End-to-End Pipeline: VietOCR + PaddleOCR + LightGBM

This notebook is strictly optimized for **Google Colab**. It downloads the dataset, installs specifically pinned versions to avoid Paddle bugs, and runs the High-Accuracy Hybrid Pipeline.

In [ ]:
# 1. Setup Kaggle Authentication & Download Dataset
!mkdir -p ~/.kaggle && echo KGAT_711b1424e8ec6f0c1114ee126af1adc9 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

!kaggle competitions download -c the-2nd-ura-hackathon
!unzip -q the-2nd-ura-hackathon.zip -d /content/dataset
print("Data downloaded and unzipped successfully!")

In [ ]:
# 2. Strict Installation of Dependencies
# Uninstall any broken paddle versions first
!pip uninstall -y paddlepaddle paddlepaddle-gpu
# Force install stable paddle 2.6.1 and PaddleOCR 2.8.1 to avoid the 'AnalysisConfig' bug
!python -m pip install paddlepaddle-gpu==2.6.1.post120 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q
!pip install paddleocr==2.8.1 vietocr lightgbm scikit-learn tqdm Pillow==9.5.0 -q
print("Dependencies installed!")

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from PIL import Image
import torch
from tqdm import tqdm
import csv

from paddleocr import PaddleOCR
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

print("Libraries loaded.")

In [ ]:
# 3. Setup Data Paths
INPUT_DIR = "/content/dataset"
TEST_CSV = f"{INPUT_DIR}/test.csv"
IMAGES_DIR = f"{INPUT_DIR}/test_images/images"

if not os.path.exists(TEST_CSV):
    print(f"Warning: {TEST_CSV} not found.")
else:
    test_df = pd.read_csv(TEST_CSV)
    print(f"Loaded {len(test_df)} test images.")

In [ ]:
# 4. Initialize Hybrid OCR Engine
class HybridOCR:
    def __init__(self):
        print("Initializing PaddleOCR Detector...")
        self.detector = PaddleOCR(use_textline_orientation=False, lang='en') 
        
        print("Initializing VietOCR Recognizer...")
        config = Cfg.load_config_from_name('vgg_transformer')
        config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        self.recognizer = Predictor(config)
        print(f"VietOCR loaded on {config['device']}")

    def process_image(self, image_path):
        try:
            result = self.detector.ocr(image_path, rec=False, cls=False)
            if not result or not result[0]:
                return ""
            
            boxes = result[0]
            boxes.sort(key=lambda x: (x[0][1], x[0][0]))
            
            img = Image.open(image_path).convert('RGB')
            texts = []
            for box in boxes:
                x_coords = [p[0] for p in box]
                y_coords = [p[1] for p in box]
                xmin, xmax = int(min(x_coords)), int(max(x_coords))
                ymin, ymax = int(min(y_coords)), int(max(y_coords))
                
                xmin, ymin = max(0, xmin - 2), max(0, ymin - 2)
                xmax, ymax = min(img.width, xmax + 2), min(img.height, ymax + 2)
                
                crop_img = img.crop((xmin, ymin, xmax, ymax))
                
                if crop_img.width > 5 and crop_img.height > 5:
                    text = self.recognizer.predict(crop_img)
                    if text:
                        texts.append(text)
            
            return " ".join(texts)
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            return ""


In [ ]:
# 5. Setup Rule-Based Extraction
BRAND_RULES = [
    (r"ha\s*long\s*canfoco.*pate.*c[ộo]t|c[ộo]t\s*đ[èe]n.*ha\s*long\s*canfoco", "Ha Long Canfoco Pate Cột Đèn", []),
    (r"ha\s*long\s*canfoco.*pate|canfoco.*pate\s*c[ộo]t|pate\s*c[ộo]t\s*đ[èe]n.*canfoco", "Ha Long Canfoco Pate", []),
    (r"ha\s*long\s*canfoco|halong\s*canfoco|canfood|canfoco", "Ha Long Canfoco", []),
    (r"đ[ồo]\s*h[ộo]p\s*h[ạa]\s*long|do\s*hop\s*ha\s*long", "Đồ Hộp Hạ Long", []),
    (r"pate\s*c[ộo]t\s*đ[èe]n|pate\s*cot\s*den|c[ộo]t\s*đ[èe]n\s*h[ảa]i\s*ph[òo]ng", "Pate Cột Đèn Hải Phòng", []),
    (r"vinamilk", "Vinamilk", ["flex", "adm gold", "sure", "canxi", "colosbaby", "ong tho", "ông thọ"]),
    (r"th true|thtrue", "TH True Milk", ["true yogurt", "grow", "school milk"]),
    (r"dutch lady|cô gái", "Dutch Lady", ["grow", "complete", "canxi"]),
    (r"nestle|nestlé", "Nestlé", ["milo", "coffee mate", "nan"]),
    (r"milo\b", "Nestlé Milo", []),
    (r"vissan", "Vissan", ["pate heo", "pate ga", "pate gà", "xuc xich", "xúc xích"]),
]

def extract_product_rule_based(text: str) -> str:
    if not text or not text.strip():
        return ""
    tl = text.lower().replace("patê", "pate")
    for pattern, brand, lines in BRAND_RULES:
        if re.search(pattern, tl, re.IGNORECASE):
            for line in lines:
                if re.search(line, tl, re.IGNORECASE):
                    return f"{brand} {line.replace('+', '+').title()}"
            return brand
    return ""


In [ ]:
# 6. Inference Loop
def run_inference():
    ocr_engine = HybridOCR()
    results = []
    
    print("Starting Inference...")
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        image_id = row['image_id']
        img_path = f"{IMAGES_DIR}/{image_id}.jpg"
        
        if os.path.exists(img_path):
            ocr_text = ocr_engine.process_image(img_path)
            product_name = extract_product_rule_based(ocr_text)
            
            results.append({
                "image_id": image_id,
                "ocr_text": ocr_text if ocr_text else " ",
                "product_name": product_name if product_name else " "
            })
        else:
            results.append({
                "image_id": image_id,
                "ocr_text": " ",
                "product_name": " "
            })
            
    submission_df = pd.DataFrame(results)
    submission_df.to_csv("submission.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
    print("Saved to submission.csv. You can now download it from the Colab file browser!")

# Run the full pipeline
run_inference()